Vamos a descargar unos ficheros grandes de una url, corresponden al servicio de taxis en Nueva York (3,4 millones de filas)
https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
descargamos los datos del taxi amarillo de enero a marzo de 2025 en formato parquet, lo pondremos en C:\PCAD
- Convertir el parquet en un csv
- Comparar el tiempo de carga entre el csv y el parquet con pandas
- Comparar el tiempo de carga entre el csv y el parquet con polars
Requisito para leer parquet debo tener instalada la libreria 
pyarrow o fastparquet >pip install pyarrow


In [ ]:
#Convertir el parquet en csv
import pandas as pd
#1. Cargar el fichero parquet a un df de pandas
df = pd.read_parquet('/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet')
#2. Obtener el numero de registros
total_registros = len(df)
#3. Guardar como .csv
df.to_csv("/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/taxis.csv", index=False)
#4. Mostar el mensaje
print("Compleato con éxito, con un total de registros de",total_registros)

In [ ]:
# Prueba de carga de los 2 ficheros tanto (.parquet como .csv) con pandas

import pandas as pd
import time
import os

# 0.path de los ficheros 

parquet_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'

csv_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/taxis.csv'

# 1. Medir el tiempo de carga del fichero parquet

start_parquet = time.time()
df_parquet = pd.read_parquet(parquet_file)
end_parquet = time.time()
tiempo_parquet = end_parquet - start_parquet

# 2. Medir el tiempo de carga del fichero CSV
start_csv = time.time()
df_csv = pd.read_csv(csv_file)
end_csv = time.time()
tiempo_csv = end_csv - start_csv

# 3. Mostrar los resultados

print(f"Tiempo de carga con parquet | pandas: {tiempo_parquet:.4f} segundos")

print(f"Tiempo de carga con CSV | pandas: {tiempo_csv :.4f} segundos")


In [ ]:
# Prueba de carga de los 2 ficheros tanto (.parquet como .csv) con polars

import polars as pl
import time
import os

# 0.path de los ficheros 

parquet_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'

csv_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/taxis.csv'

# 1. Medir el tiempo de carga del fichero parquet
start_parquet = time.time()
df_parquet = pl.read_parquet(parquet_file)
end_parquet = time.time()
tiempo_parquet = end_parquet - start_parquet

# 2. Medir el tiempo de carga del fichero CSV
start_csv = time.time()
df_csv = pl.read_csv(csv_file)
end_csv = time.time()
tiempo_csv = end_csv - start_csv

# 3. Mostrar los resultados
print(f"Tiempo de carga con parquet y polars: {tiempo_parquet:.4f} segundos")

print(f"Tiempo de carga con CSV y polars: {tiempo_csv :.4f} segundos")


In [ ]:
# número de registros
len(df_parquet)

In [ ]:
# ver los primeros 50 registros
df_parquet.head(50)

In [ ]:
# ver los campos de un fichero sin cargarlo (Con polars)

import polars as pl

# 0.path de los ficheros 

parquet_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'

# 1. Cargar el esquema del conjunto de datos
esquema = pl.read_parquet_schema(parquet_file)

# 2. Ver el resultado
#print(esquema)     # Lo muestra todo seguido
esquema             # Se ve mas claro por filas

Con polars podremos trabajar en 2 modos, modo eager (inmediato), o en modo Lazy (Plan de ejecución)

Modo eager paso a paso:

- Carga fichero
- Filtra
- crea un dataframe temporal
- agrupa
- crea otro dataframe temporal
- calcula la métrica (sum, avg, max, min, count)
- ordena
- devuelve resultado

Cada linea ejecuta el trabajo de forma inmediata (sin optimizacion global) 




In [ ]:
# Con este fichero taxi enero del parquet vamos a calcular la propina media y el ticket medio (el pago) por número de passajeros 
# (en NYC, el número máximo de pasajeros creo que son 9) , modo eager

import polars as pl

# 0.path de los ficheros 

parquet_file = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'

# 1.  carga inmediata en modo eager

df_taxi_01 = pl.read_parquet(parquet_file)

# 2. Comprobar si se ha cargado bien, visualizamos los 20 primeros
df_taxi_01.head(20)

# 3. VAmos a filtrar + agrupar + agregar + ordenar

resultado = (
    df_taxi_01
    .filter(
        (pl.col('trip_distance')>1) & 
        (pl.col('total_amount')>0) &
        (pl.col('passenger_count').is_not_null())
    )
    .group_by('passenger_count')
    .agg(
        pl.len().alias('Numero viajes'), # numero de viajes
        pl.col('trip_distance').mean().round(2).alias('distancia media'),   # Distancia media
        pl.col('total_amount').mean().round(2).alias('tiquet ,medio'),      # El precio medio viaje
        pl.col('tip_amount').mean().round(2).alias('propina media')        # La propina media
    )
    .sort('passenger_count', descending=True)
)

resultado

- Polars permite concat igual que pandas
- Nombres de campos a usar:
* passenger_count   = numero de pasajeros
* trip_distance     = Distancia trayecto
* fare_amount       = Tarifa base
* total_amount      = Total_pagado
* DOLocationID      = Zona de destino

PRACTICA 4 : (High Level) : 
Necesitamos averiguar el número de viajes, el precio medio de total_amount, el precio medio de fare_amount por zona de destino, siempre y cuando el número de pasajetos sea mayor que 2, la distancia del trayecto mayor de 2.5 millas y que el fare_amount sea mayor de 20. De los meses de enero a marzo del 2025 y medir el tiempo total de proceso desde la carga hasta el resultado

In [22]:

import polars as pl
import time

# 0.path de los ficheros 

parquet_file01 = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'
parquet_file02 = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-02.parquet'
parquet_file03 = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-03.parquet'

# 1.  carga inmediata en modo eager

start_time = time.time()

df_taxi_01 = pl.read_parquet(parquet_file01)
df_taxi_02 = pl.read_parquet(parquet_file02)
df_taxi_03 = pl.read_parquet(parquet_file03)

# 3. omprobar la carga de los 3 primeros de cada taxi dfs

print(df_taxi_01.head(3))
print(df_taxi_02.head(3))
print(df_taxi_03.head(3))

df_taxi_1T = pl.concat([df_taxi_01, df_taxi_02, df_taxi_03])

# ver el numero de registro
print(len(df_taxi_1T))

resultado = (
    df_taxi_1T
    .filter(
        (pl.col('passenger_count')>2) & 
        (pl.col('trip_distance')>2.5) &
        (pl.col('fare_amount')>20)
    )
    .group_by('DOLocationID')
    .agg(
        pl.len().alias('Numero viajes'),                                # numero de viajes
        pl.col('total_amount').mean().round(2).alias('Promedio del total'),   # El precio medio viaje      
        pl.col('fare_amount').mean().round(2).alias('Promedio tarifa base'), # Distancia media
    )
    .sort('Promedio del total')
)

end_time = time.time()
tiempo_total = end_time - start_time

print (f"el tiempo total del proceso es: {tiempo_total:.4f} segundos")
resultado


shape: (3, 20)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ VendorID ┆ tpep_pick ┆ tpep_drop ┆ passenger ┆ … ┆ total_amo ┆ congestio ┆ Airport_f ┆ cbd_conge │
│ ---      ┆ up_dateti ┆ off_datet ┆ _count    ┆   ┆ unt       ┆ n_surchar ┆ ee        ┆ stion_fee │
│ i32      ┆ me        ┆ ime       ┆ ---       ┆   ┆ ---       ┆ ge        ┆ ---       ┆ ---       │
│          ┆ ---       ┆ ---       ┆ i64       ┆   ┆ f64       ┆ ---       ┆ f64       ┆ f64       │
│          ┆ datetime[ ┆ datetime[ ┆           ┆   ┆           ┆ f64       ┆           ┆           │
│          ┆ μs]       ┆ μs]       ┆           ┆   ┆           ┆           ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1        ┆ 2025-01-0 ┆ 2025-01-0 ┆ 1         ┆ … ┆ 18.0      ┆ 2.5       ┆ 0.0       ┆ 0.0       │
│          ┆ 1         ┆ 1         ┆           ┆   ┆           ┆           ┆

DOLocationID,Numero viajes,Promedio del total,Promedio tarifa base
i32,u32,f64,f64
203,79,34.95,28.22
124,124,35.36,28.22
205,110,36.77,29.75
12,558,37.42,28.73
130,203,39.03,31.97
…,…,…,…
118,10,139.38,101.98
84,4,142.73,123.6
204,1,150.03,130.4


In [ ]:
# Solucion practica 4 en (Modo Lazy - con optimizacion)
# vamos a partir del parquer de enero con el trimestre de (enero a marzo)

# En modo Lazy no cargamos o ejecutamos acciones hasta la instruccion collect

# 1. por tanto no haremo read, si no scan

import polars as pl
import os
import time

# Ruta de la carpeta

carpeta_parquet = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/'

consulta = (
    pl.scan_parquet(os.path.join(carpeta_parquet, 
                                 "yellow_tripdata_2025-0[1-3].parquet"))
    .filter(
        (pl.col('passenger_count')>2) & 
        (pl.col('trip_distance')>2.5) &
        (pl.col('fare_amount')>20)
    )
    .group_by('DOLocationID')
    .agg(
        pl.len().alias('Numero viajes'),                                # numero de viajes
        pl.col('total_amount').mean().round(2).alias('Promedio del total'),   # El precio medio viaje      
        pl.col('fare_amount').mean().round(2).alias('Promedio tarifa base'), # Distancia media
    )
    .sort('Promedio del total')
)

# 2. Mostrar el resultado
print(consulta.explain())

# 3. ¿Como ejecutar?

resultado_lazy = consulta.collect()

4.# imprimir resultado

resultado_lazy

In [21]:
# Solucion practica 4 en (Modo Lazy con los 3 meses - con optimizacion)
# vamos a partir del parquer de enero con el trimestre de (enero a marzo)

# En modo Lazy no cargamos o ejecutamos acciones hasta la instruccion collect

# 1. por tanto no haremo read, si no scan

import polars as pl

parquet_file01 = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet'

start_time = time.time()

consulta = (
    pl.scan_parquet(parquet_file01)
    .filter(
        (pl.col('passenger_count')>2) & 
        (pl.col('trip_distance')>2.5) &
        (pl.col('fare_amount')>20)
    )
    .group_by('DOLocationID')
    .agg(
        pl.len().alias('Numero viajes'),                                # numero de viajes
        pl.col('total_amount').mean().round(2).alias('Promedio del total'),   # El precio medio viaje      
        pl.col('fare_amount').mean().round(2).alias('Promedio tarifa base'), # Distancia media
    )
    .sort('Promedio del total')
)

# 2. Mostrar el resultado
print(consulta.explain())

# 3. ¿Como ejecutar?

resultado_lazy = consulta.collect()

end_time = time.time()
tiempo_total = end_time - start_time

print (f"el tiempo total del proceso es: {tiempo_total:.4f} segundos")

4.# imprimir resultado

resultado_lazy

SORT BY [col("Promedio del total")]
  AGGREGATE[maintain_order: false]
    [len().alias("Numero viajes"), col("total_amount").mean().round().alias("Promedio del total"), col("fare_amount").mean().round().alias("Promedio tarifa base")] BY [col("DOLocationID")]
    FROM
    simple π 3/3 ["total_amount", "fare_amount", ... 1 other column]
      Parquet SCAN [/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/yellow_tripdata_2025-01.parquet]
      PROJECT 5/20 COLUMNS
      SELECTION: [([([(col("passenger_count")) > (2)]) & ([(col("trip_distance")) > (2.5)])]) & ([(col("fare_amount")) > (20.0)])]
      ESTIMATED ROWS: 3475226
el tiempo total del proceso es: 0.0296 segundos


DOLocationID,Numero viajes,Promedio del total,Promedio tarifa base
i32,u32,f64,f64
218,14,31.71,25.51
203,34,35.0,27.73
222,5,35.83,29.74
124,42,35.87,28.23
12,171,35.97,27.92
…,…,…,…
176,3,134.96,113.83
23,4,141.03,114.32
84,2,147.34,119.9


Vamos a cargar unos dataframes partiendo de un fichero de excel y despues de calcular nla edad y la decada haremos merge

In [23]:
# El objetivo de la practica es trabajar sobre el merge, sobre el excel de telefonia 
# cargamos los primeros 3 meses (ene-mar) -
# tambien cargaremos la hoja franja edades. Franja_Edades
import pandas as pd

ruta_fichero = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Datos Telefonia Separados Meses Comerciales URL.xlsx'
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
franjas = pd.read_excel(ruta_fichero,sheet_name="Franja_Edades", header=0)

#Concatenar los 3 meses
fras_trimestre = pd.concat([fras_enero,fras_febrero,fras_marzo],ignore_index=True)

#Eliminar las filas en blanco
fras_trimestre = fras_trimestre.dropna(how='all')

# Vamos a calcualr la edad + la decada, que ya la teníamos del otro ejercicio Del dia 2026-04-30 _ Practica 3.2

fras_trimestre['Dias'] = (pd.Timestamp('today') - fras_trimestre['Fecha Nacimiento']).dt.days
fras_trimestre['Edad'] = fras_trimestre['Dias'] // 365

# Muestrame el dataframe

fras_trimestre


,Nombre,Servicios,Genero,Fecha Nacimiento,Localidad,Fecha factura,Importe factura,Satisfacción,IdComercial,Dias,Edad
0,Cliente 1,Móvil,Masculino,1969-12-24,Madrid,2020-01-01,92.0,10.0,6.0,20585,56
1,Cliente 10,Voz IP,Femenino,1989-06-26,Baleares,2020-01-01,8.0,4.0,9.0,13461,36
2,Cliente 100,ADSL,Masculino,1989-09-23,Madrid,2020-01-01,49.0,7.0,3.0,13372,36
3,Cliente 1000,Móvil,Masculino,1950-10-16,Madrid,2020-01-01,25.0,7.0,7.0,27594,75
4,Cliente 1001,Voz IP,Masculino,1966-09-18,Castilla La Mancha,2020-01-01,40.0,4.0,3.0,21778,59
...,...,...,...,...,...,...,...,...,...,...,...
6169,Cliente 2008,Fibra,Femenino,1992-04-13,Barcelona,2020-03-01,65.0,3.0,3.0,12439,34
6170,Cliente 2009,ADSL,Masculino,1963-06-29,Barcelona,2020-03-01,71.0,7.0,5.0,22955,62
6171,Cliente 2010,Móvil + Fijo,Masculino,1976-01-17,Barcelona,2020-03-01,49.0,2.0,7.0,18370,50
6172,Cliente 2011,Fibra,Femenino,1982-07-10,Barcelona,2020-03-01,53.0,8.0,2.0,16004,43
